<a href="https://colab.research.google.com/github/jessicalewinter/deep-learning-specialization/blob/main/course-notebooks/observability/Aula02_Exercicio_Churn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 2 · Exercício: comparando modelos de churn com o W&B

**Observabilidade de Modelos e Prototipagem de Aplicações IA**

Voltamos ao problema do **churn** (cancelamento de clientes) para praticar o que vimos na aula: registrar experimentos no W&B e **comparar runs** usando as métricas certas.

- **Parte 1 (em aula):** rodar quatro experimentos já definidos e comparar os resultados no W&B.
- **Parte 2 (para casa, individual):** fazer seus próprios experimentos para escolher o modelo que você levaria para a equipe de retenção.

Este notebook não precisa de GPU.

In [1]:
!pip install -q wandb

## 1. Preparação dos dados

Usamos o dataset **Telco Customer Churn original**. A função abaixo prepara os dados para você: transforma as colunas categóricas em números e separa 80% para treino e 20% para validação, mantendo a mesma proporção de cancelamentos nos dois conjuntos.

> A coluna `TotalCharges` ficou de fora: ela é quase igual a `tenure × MonthlyCharges`, e voltaremos aos detalhes dela na discussão do exercício da aula 1.

In [2]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.model_selection import train_test_split

URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

def preparar_dados():
    dados = pd.read_csv(URL).drop(columns=["customerID", "TotalCharges"])
    y = dados.pop("Churn").eq("Yes").astype(int)               # 1 = cancelou, 0 = ficou
    X = pd.get_dummies(dados, drop_first=True).astype(float)   # categorias viram colunas 0/1
    return train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

X_tr, X_va, y_tr, y_va = preparar_dados()
print(f"Treino: {len(X_tr)} clientes · Validação: {len(X_va)} clientes · {X_tr.shape[1]} colunas")
print(f"Taxa de churn na validação: {y_va.mean():.1%}")

Treino: 5634 clientes · Validação: 1409 clientes · 29 colunas
Taxa de churn na validação: 26.5%


## 2. A função de experimento

Cada chamada de `experimento` cria **um run** no W&B: treina o modelo, calcula as métricas na validação e registra tudo. As linhas que falam com o W&B estão marcadas com `# <<< W&B`.

O **limiar** é a probabilidade a partir da qual o modelo diz "vai cancelar". O padrão é 0,5.

In [ ]:
import wandb
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

wandb.login()
PROJETO = "aula2-churn"

def experimento(modelo, nome, config, limiar=0.5):
    run = wandb.init(project=PROJETO, name=nome,                    # <<< W&B: um run por experimento
                     config={**config, "limiar": limiar})          # <<< W&B: a configuração inclui o limiar

    modelo.fit(X_tr, y_tr)                                          # treina (aqui, um evento rápido)
    prob = modelo.predict_proba(X_va)[:, 1]                         # probabilidade de cancelar
    pred = (prob >= limiar).astype(int)                             # decisão, aplicando o limiar

    metricas = {
        "acuracia": accuracy_score(y_va, pred),
        "precisao": precision_score(y_va, pred),
        "recall": recall_score(y_va, pred),
        "f1": f1_score(y_va, pred),
    }
    run.log(metricas)                                               # <<< W&B: as quatro métricas do run

    # Matriz de confusão no mesmo formato dos slides: linhas = real, colunas = previsto
    mc = confusion_matrix(y_va, pred, labels=[1, 0])
    fig = px.imshow(mc, text_auto=True, color_continuous_scale="Blues",
                    x=["Previsto: Cancela", "Previsto: Fica"], y=["Real: Cancela", "Real: Fica"],
                    labels={"color": "clientes"}, title=f"Matriz de confusão · {nome}")
    run.log({"matriz_confusao": fig})                               # <<< W&B: figura plotly

    run.finish()                                                    # <<< W&B: fecha o run
    print(nome, {k: round(v, 3) for k, v in metricas.items()})
    return modelo

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


---
# Parte 1 (em aula): replicar e comparar

Rode os quatro experimentos abaixo. Repare que o último é **o mesmo modelo** do primeiro, só com outro limiar.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

experimento(make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
            "logistica", {"modelo": "regressao_logistica"})

experimento(DecisionTreeClassifier(max_depth=5, random_state=0),
            "arvore_prof5", {"modelo": "arvore", "max_depth": 5})

experimento(RandomForestClassifier(n_estimators=200, random_state=0),
            "floresta", {"modelo": "floresta", "n_estimators": 200})

experimento(make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
            "logistica_limiar0.3", {"modelo": "regressao_logistica"}, limiar=0.3)

👉 No projeto `aula2-churn` do W&B:
1. Abra a **tabela de runs** (ícone de tabela, à esquerda) e ordene os runs por `acuracia`, depois por `recall` e por `f1`.
2. Crie um painel de **barras** (*Add panel → Bar chart*) para cada métrica, com um run por barra.
3. Abra as **matrizes de confusão** dos runs `logistica` e `logistica_limiar0.3` lado a lado.

**Para discutir:**
- Qual é o melhor modelo pela acurácia? E pelo recall? E pelo F1? A ordem muda?
- O que mudou entre `logistica` e `logistica_limiar0.3`, olhando a matriz de confusão? Quantos falsos negativos a mais ou a menos?
- Lembrando do slide "qual erro custa mais?": qual desses quatro você levaria para a equipe de retenção?

---
# Parte 2 (para casa): encontre o seu modelo

A equipe de retenção definiu uma regra: **as ligações só valem a pena se pelo menos metade dos clientes apontados for de fato cancelar**, ou seja, precisão de pelo menos 0,50. Dentro dessa regra, eles querem encontrar o maior número possível de clientes que vão cancelar: o **maior recall**.

Faça **pelo menos 6 experimentos novos** para encontrar o melhor modelo dentro dessa regra. Alguns caminhos:
- **Limiar:** testar outros valores (0,2, 0,25, 0,35, 0,4...), também nos outros modelos;
- **Desbalanceamento:** `class_weight="balanced"` na regressão logística, na árvore ou na floresta, que dá mais peso aos clientes que cancelam;
- **Hiperparâmetros:** `max_depth` da árvore, `C` da regressão logística, `n_estimators` e `min_samples_leaf` da floresta.

Use nomes de run e configurações claros: é isso que vai permitir comparar os experimentos no W&B.

**Entregue (individual, até a véspera da aula 3):**
1. este notebook com os seus experimentos;
2. um texto curto respondendo: qual modelo você escolheu, e quais painéis do W&B justificam a escolha? Inclua capturas de tela da tabela de runs e dos painéis que usou.

In [ ]:
# Exemplos de partida (descomente e adapte)

# experimento(make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced")),
#             "logistica_balanceada", {"modelo": "regressao_logistica", "class_weight": "balanced"})

# experimento(RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=0),
#             "floresta_folha5_limiar0.35", {"modelo": "floresta", "n_estimators": 200, "min_samples_leaf": 5},
#             limiar=0.35)

In [ ]:
# Seus experimentos aqui

### Minha escolha

- **Modelo escolhido:**
- **Precisão e recall na validação:**
- **Por que este modelo (quais painéis mostram isso):**
- **Capturas de tela:**